# 02 - APT Simulation Analysis

Skeleton notebook for running and analysing ATT&CK coverage simulations against
threat-group technique sets. Drives the Phase-4 simulation engine
(`analysis.coverage_analyzer` + the `simulate_apt_attack` pipeline) and surfaces
per-group coverage, rollups, and acquisition recommendations.

**Contents**

1. Environment bootstrap - reuse the `src/` + `DATABASE_URL` setup from `01`
2. Configuration probe - load `config.yaml` and show the active simulation knobs
3. Group picker - resolve a group id (G0016/G0096/G0032) and preview its technique set
4. Run simulation - call `simulate_apt_attack` and render the `SimulationResult`
5. Per-tactic coverage - detected vs missed, with the rollup accounting
6. Recommendations - greedy marginal-gain suggestions from `generate_recommendations`
7. Multi-group comparison - re-run across the three golden groups and compare

Cells ship un-executed. Run All once the coverage engine has populated
`technique_coverage` and the simulation module is importable. Golden expectations
live in `research/p4-wave1-full-results.md`.

In [ ]:
# --- Bootstrap ----------------------------------------------------------------
# Mirror 01_attack_exploration: make `src/` importable and default DATABASE_URL
# to the repo-local SQLite file. Idempotent: safe to re-run.
import os
import sys
from pathlib import Path

_cwd = Path.cwd()
REPO_ROOT = _cwd if (_cwd / 'src').is_dir() else _cwd.parent
if not (REPO_ROOT / 'src').is_dir():
    raise RuntimeError(
        f'Could not locate repo root containing src/ (started from {_cwd}); '
        'launch Jupyter from the repository root or from notebooks/'
    )

_SRC = str(REPO_ROOT / 'src')
if _SRC not in sys.path:
    sys.path.insert(0, _SRC)

os.environ.setdefault(
    'DATABASE_URL',
    f"sqlite:///{(REPO_ROOT / 'db' / 'coverage.db').as_posix()}",
)

_db_url = os.environ['DATABASE_URL']
print(f'repo root    : {REPO_ROOT}')
print(f'DATABASE_URL : {_db_url}')

## Imports and engine

Centralise the imports here so later cells stay short. The simulation engine
module is guarded: if the Wave-2 `simulate_apt_attack` symbol is not yet
importable, degrade gracefully instead of failing the whole notebook.

In [ ]:
# --- Imports & engine ---------------------------------------------------------
import importlib.util
import logging

import pandas as pd
import plotly.express as px
from sqlalchemy import inspect, select

from config_loader import load_config
from database.models import (
    AptGroup,
    AptTechnique,
    SimulationResult,
    SimulationTechnique,
    Technique,
)
from database.session import make_engine, session_scope

engine = make_engine()
_inspector = inspect(engine)

# Probe for the simulation engine without hard-failing if Wave-2 is mid-merge.
sim_engine = None
for _mod in ('analysis.simulation', 'analysis.simulate', 'analysis.simulation_engine'):
    if importlib.util.find_spec(_mod) is not None:
        _candidate = importlib.import_module(_mod)
        if all(hasattr(_candidate, _a) for _a in ('simulate_apt_attack', 'generate_recommendations')):
            sim_engine = _candidate
            break

_has_sim = sim_engine is not None
print('simulation engine:', sim_engine.__name__ if _has_sim else 'NOT IMPORTABLE')
if not _has_sim:
    print(
        'WARNING: simulate_apt_attack / generate_recommendations not found; '
        'run cells below once the Wave-2 simulation module lands.'
    )

## Configuration probe

Load `config.yaml` and display the knobs the simulation consumes: platform scope,
confidence threshold, and the enabled log sources. The engine treats config as the
single source of truth for source enablement, so what you see here is exactly what
the simulation will use.

In [ ]:
# --- Configuration ------------------------------------------------------------
config = load_config(str(REPO_ROOT / 'config.yaml'))

print('platform_scope             :', config['platform_scope'])
print('detection_confidence_thresh:', config['detection_confidence_threshold'])
print('enabled_log_sources        :')
for _fmt, _on in sorted(config['enabled_log_sources'].items()):
    print(f'  {_fmt:<16} {"ON" if _on else "off"}')

## Group picker
Resolve a group by its `group_id` (e.g. `G0016`) and preview the technique set the
simulation will score. Unknown ids resolve to a `ValueError` listing valid ids, so
this cell also sanity-checks the id before we run anything.

In [ ]:
# --- Group picker -------------------------------------------------------------
GROUP_ID = 'G0016'  # change to G0096 / G0032 for the other golden groups

with session_scope(engine) as session:
    group = session.scalars(
        select(AptGroup).where(AptGroup.group_id == GROUP_ID)
    ).first()

if group is None:
    raise ValueError(f'unknown group_id {GROUP_ID!r}')

print('group      :', group.group_id, '-', group.name)
print('stix_id    :', group.stix_id)
print('aliases    :', group.aliases)

with session_scope(engine) as session:
    tech_rows = session.execute(
        select(
            Technique.attack_id,
            Technique.name,
            Technique.tactics,
            Technique.is_subtechnique,
            Technique.revoked,
        )
        .join(AptTechnique, AptTechnique.technique_stix_id == Technique.stix_id)
        .where(
            AptTechnique.group_stix_id == group.stix_id,
            Technique.revoked.is_(False),
        )
        .order_by(Technique.attack_id)
    ).all()

print(f'\nalive techniques: {len(tech_rows)}')
print(
    'sub-techniques :',
    sum(1 for r in tech_rows if r.is_subtechnique),
)
preview_df = pd.DataFrame(
    [
        {
            'attack_id': r.attack_id,
            'name': r.name,
            'tactics': ', '.join(r.tactics or []),
            'is_subtechnique': r.is_subtechnique,
        }
        for r in tech_rows
    ]
)
display(preview_df.head(15))
if len(preview_df) > 15:
    print(f'... ({len(preview_df) - 15} more)')

## Run simulation
Invoke `simulate_apt_attack(group_id, session, persist=True)`. The engine resolves
the group, scores every alive technique against the enabled-datasource coverage
rows, classifies each as `detected` / `detected_rollup` / `missed`, persists a
`SimulationResult` + one `SimulationTechnique` per technique, and returns the
result. `coverage_pct = detected / total` (0.0-guarded, rounded to 4 decimals).

The invariant `detected + detected_rollup + missed == total` is asserted below.

In [ ]:
# --- Run simulation -----------------------------------------------------------
if not _has_sim:
    raise RuntimeError('simulation engine not importable - see Imports cell')

with session_scope(engine) as session:
    result = sim_engine.simulate_apt_attack(GROUP_ID, session, persist=True)

print('simulation id     :', result.id)
print('group_id          :', result.group_id)
print('run_at            :', result.run_at)
print('platform_scope    :', result.platform_scope)
print('threshold         :', result.confidence_threshold)
print('total_techniques  :', result.total_techniques)
print('detected          :', result.detected_count)
print('missed            :', result.missed_count)
print('coverage_pct      :', result.coverage_pct)
print('status            :', result.status)

# Invariant check: detected + detected_rollup + missed == total.
_summary = result.summary or {}
_counts = _summary.get('counts', {})
_rollup = _counts.get('detected_rollup', 0)
_total = result.detected_count + _rollup + result.missed_count
_msg = (
    'rollup invariant broken: '
    + str(result.detected_count) + '+' + str(_rollup) + '+'
    + str(result.missed_count) + ' != ' + str(result.total_techniques)
)
assert _total == result.total_techniques, _msg
print(
    '\ninvariant OK: '
    + str(result.detected_count) + ' detected + '
    + str(_rollup) + ' rollup + '
    + str(result.missed_count) + ' missed = '
    + str(result.total_techniques)
)

## Per-tactic coverage
Break the result down by tactic using the persisted `SimulationTechnique` rows.
A technique counts toward a tactic if the tactic appears in its `tactic` field;
multi-tactic techniques are attributed to each tactic they carry. The bar chart
stacks detected (incl. rollup) against missed.

In [ ]:
# --- Per-tactic coverage ------------------------------------------------------
with session_scope(engine) as session:
    sim_techniques = session.scalars(
        select(SimulationTechnique).where(
            SimulationTechnique.result_id == result.id
        )
    ).all()

if not sim_techniques:
    print('No simulation_techniques persisted for this run.')
else:
    _rows = [
        {
            'attack_id': st.technique_attack_id,
            'tactic': st.tactic,
            'status': st.status,
            'confidence': st.confidence,
        }
        for st in sim_techniques
    ]
    st_df = pd.DataFrame(_rows)

    tally = (
        st_df.groupby(['tactic', 'status'])
        .size()
        .unstack(fill_value=0)
        .reindex(columns=['detected', 'detected_rollup', 'missed'], fill_value=0)
    )
    tally['total'] = tally.sum(axis=1)
    tally['detected_total'] = tally['detected'] + tally['detected_rollup']
    display(tally)

    fig_tactic = px.bar(
        tally.reset_index(),
        x='tactic',
        y=['detected_total', 'missed'],
        title=f'Per-tactic coverage for {GROUP_ID} ({result.group_id})',
        labels={'value': 'techniques', 'variable': 'status'},
        barmode='stack',
    )
    fig_tactic.show()

## Recommendations
Call `generate_recommendations(result, session, top_n=3)`. The engine ranks
*disabled* log sources by how many currently-missed techniques each would close
(greedy marginal gain), tie-broken by source priority, and excludes anything
already enabled. Templates:

- `Enable {source_label} to cover {n} missed techniques ({ids_truncated})`
- `No configured source maps to {tid} - add a mapping row.` (fallback)

In [ ]:
# --- Recommendations ----------------------------------------------------------
with session_scope(engine) as session:
    recs = sim_engine.generate_recommendations(result, session, top_n=3)

if not recs:
    print('No actionable recommendations (all sources enabled or no closable gaps).')
else:
    for i, rec in enumerate(recs, 1):
        print(f'{i}. {rec}')

## Multi-group comparison
Re-run the simulation across the three golden groups (G0016=66, G0096=82, G0032=93
techniques) under the same config and compare coverage. Determinism is guaranteed
by the engine (sorted iteration, rounded floats, no set-order dependence), so
re-runs produce identical `coverage_pct` values.

In [ ]:
# --- Multi-group comparison ---------------------------------------------------
GOLDEN_GROUPS = ['G0016', 'G0096', 'G0032']

if not _has_sim:
    print('simulation engine not importable - skipping comparison')
    compare_df = pd.DataFrame()
else:
    _compare_rows = []
    for _gid in GOLDEN_GROUPS:
        with session_scope(engine) as session:
            _res = sim_engine.simulate_apt_attack(_gid, session, persist=True)
        _compare_rows.append(
            {
                'group_id': _res.group_id,
                'total': _res.total_techniques,
                'detected': _res.detected_count,
                'detected_rollup': (
                    (_res.summary or {}).get('counts', {}).get('detected_rollup', 0)
                ),
                'missed': _res.missed_count,
                'coverage_pct': _res.coverage_pct,
            }
        )
    compare_df = pd.DataFrame(_compare_rows)
    display(compare_df)

    fig_compare = px.bar(
        compare_df,
        x='group_id',
        y='coverage_pct',
        title='Coverage comparison across golden groups',
        labels={'coverage_pct': 'coverage (detected / total)'},
        text='coverage_pct',
    )
    fig_compare.update_traces(texttemplate='%{text:.4f}', textposition='outside')
    fig_compare.show()

## Notes and next steps

- Golden numbers (default config): G0016=66, G0096=82, G0032=93 techniques; ~24
  techniques detectable under the default windows/0.6 config. evtx dominates;
  zeek covers the network layer (T1046/T1048/T1071.004/T1496/T1571); aws_cloudtrail
  covers T1110/T1531/T1685.
- Rollups: sub-technique -> parent when the parent is detectable (e.g. G0016
  T1685.002 -> T1685, T1078 -> T1078.003); parent -> children when any child is
  detectable (e.g. G0096 T1496.001 -> T1496).
- Each run persists a fresh `SimulationResult` + `SimulationTechnique` rows; the
  table grows over time. Truncate via a direct SQL DELETE if you need a clean slate.
- Compare against `research/p4-wave1-full-results.md` for the Wave-1 baseline.
- Next: feed the per-tactic gaps into `gap_clustering` to explain *why* the
  undetected techniques cluster.